# Session 8 — PDF RAG with FAISS

This notebook builds a minimal retrieval-augmented generation pipeline over generic handbook and benefits PDFs.

## Learning Goals

- understand why PDFs need preprocessing
- inspect extracted document text
- chunk documents with overlap
- build a FAISS index over embeddings
- compare no-RAG vs RAG answers

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()

current = Path.cwd().resolve()
if (current / 'material').exists():
    repo_root = current
elif (current.parent / 'material').exists():
    repo_root = current.parent
else:
    repo_root = current.parent.parent

session_dir = repo_root / 'material' / 'Session 8'
helpers_dir = session_dir / 'helpers'
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from pdf_utils import extract_pdf_text, join_pages
from rag_utils import chunk_text, package_chunks, build_faiss_index, search_index, build_grounded_prompt

data_dir = session_dir / 'data'
pdf_dir = data_dir / 'pdfs'
questions_path = data_dir / 'questions.json'

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
print('repo_root =', repo_root)
print('pdf_count =', len(list(pdf_dir.glob('*.pdf'))))

## 1. Load the PDF Corpus

In [ ]:
pdf_paths = sorted(pdf_dir.glob('*.pdf'))
for path in pdf_paths:
    print(path.name)

## 2. Extract Text

PDFs are not directly useful to an LLM. We first need to extract text from each page.

In [ ]:
documents = []
for pdf_path in pdf_paths:
    pages = extract_pdf_text(pdf_path)
    full_text = join_pages(pages)
    documents.append({
        'source_file': pdf_path.name,
        'pages': pages,
        'text': full_text,
    })

for doc in documents:
    print(doc['source_file'], 'pages =', len(doc['pages']), 'chars =', len(doc['text']))

In [ ]:
sample_doc = documents[0]
print(sample_doc['source_file'])
print(sample_doc['text'][:1200])

## 3. Chunk the Documents

Chunking breaks long text into smaller pieces. Overlap helps preserve context across chunk boundaries.

In [ ]:
all_chunks = []
for doc in documents:
    chunks = chunk_text(doc['text'], chunk_size=500, overlap=100)
    all_chunks.extend(package_chunks(chunks, doc['source_file']))

print('total chunks =', len(all_chunks))
all_chunks[:2]

## 4. Create Embeddings

Session 8 is OpenAI-first, so the preferred path uses OpenAI embeddings. A local sentence-transformers fallback is included for offline work.

In [ ]:
chunk_texts = [chunk['text'] for chunk in all_chunks]

if OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    embedding_response = client.embeddings.create(
        model='text-embedding-3-small',
        input=chunk_texts
    )
    chunk_embeddings = np.array([item.embedding for item in embedding_response.data], dtype='float32')
    embedding_source = 'OpenAI embeddings'
else:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    chunk_embeddings = embedding_model.encode(chunk_texts, convert_to_numpy=True).astype('float32')
    embedding_source = 'sentence-transformers fallback'

print(embedding_source)
print(chunk_embeddings.shape)

## 5. Build a FAISS Index

In [ ]:
index = build_faiss_index(chunk_embeddings)
print(index.ntotal)

## 6. Retrieve Relevant Chunks

In [ ]:
question = 'Which health plan appears to offer broader benefits coverage?'

if OPENAI_API_KEY:
    query_embedding = np.array(
        OpenAI(api_key=OPENAI_API_KEY).embeddings.create(
            model='text-embedding-3-small',
            input=[question]
        ).data[0].embedding,
        dtype='float32'
    )
else:
    query_embedding = embedding_model.encode([question], convert_to_numpy=True).astype('float32')[0]

distances, indices = search_index(index, query_embedding, top_k=3)
retrieved_chunks = [all_chunks[idx] for idx in indices[0]]
retrieved_chunks

In [ ]:
for chunk in retrieved_chunks:
    print(f"[{chunk['source_file']} | chunk {chunk['chunk_id']}]")
    print(chunk['text'][:700])
    print('-' * 80)

## 7. Build a Grounded Prompt

In [ ]:
grounded_prompt = build_grounded_prompt(question, retrieved_chunks)
print(grounded_prompt[:1800])

## 8. Compare No-RAG vs RAG

These cells are optional if you do not have live API access. The comparison is most useful with a real model.

In [ ]:
with open(questions_path, 'r', encoding='utf-8') as f:
    questions = json.load(f)

questions[:3]

In [ ]:
# Requires OPENAI_API_KEY for live generation.
# Skip this cell if you do not have live API access.

client = OpenAI(api_key=OPENAI_API_KEY)

demo_question = questions[0]['question']

no_rag = client.responses.create(
    model='gpt-4o-mini',
    instructions='Answer concisely.',
    input=demo_question,
)

rag = client.responses.create(
    model='gpt-4o-mini',
    instructions='Use the provided context only. If the answer is not in the context, say so.',
    input=grounded_prompt,
)

print('NO RAG:')
print(no_rag.output_text)
print('\nRAG:')
print(rag.output_text)

## 9. Extensions

- **OpenAI Vector Store:** managed alternative when you want OpenAI to handle chunking, embedding, and indexing.
- **Hybrid search:** production systems often combine keyword and dense retrieval.
- **Agentic RAG:** in Session 9, retrieval can become a tool inside an agent loop rather than a one-shot pipeline.